## This script uses Russell (2022) TIMPS data (https://home.chpc.utah.edu/~mani/TIMPS_v2/) to calculate and plot storm-relative motion (SRM) vectors for each sonde from each CAMP2Ex convective case that is a TIMPS-tracked MCS.

In [ ]:
import os
import sys
import math
import h5py
import xarray as xr
import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm  #to get python's normal library of colormaps
import matplotlib.colors as mplc

import cartopy.crs as ccrs
import cartopy.feature as cfeature
#from cartopy.util import add_cyclic_point
#from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

from datetime import datetime
from datetime import timedelta

import metpy.calc as mpcalc
from metpy.units import units

from PIL import Image
import icartt            #needed to read .ict files

import time

import warnings
warnings.filterwarnings("ignore")  #hides "UserWarning" with icartt files


In [ ]:
timps_test_path = os.path.join(os.getcwd(), 'TIMPS_data', 'fields2D_precip_n_mcs', '201909', 'Tracking_201909170300.nc')

testds = xr.open_dataset(timps_test_path)
testds


In [ ]:
timps_test_path2 = os.path.join(os.getcwd(), 'TIMPS_data', 'TIMPS_files', 'TIMPS_1646682_201908300000_08_-149.nc')

testds2 = xr.open_dataset(timps_test_path2)
testds2


In [ ]:
#Ignore this cell for CAMP2Ex (this cell was for CPEX-CV)

# #To calculate total mean propagation speed and direction for a given TIMPS-tracked MCS:
# timps_test_path2 = os.path.join(os.getcwd(), 'TIMPS_data', 'propvecw_mag_Sept2022.nc')  #download from https://www.inscc.utah.edu/~mani/cpex_cv/lifetime_prop/2022/09/
# timps_test_path3 = os.path.join(os.getcwd(), 'TIMPS_data', 'propvecw_dir_Sept2022.nc')  #download from https://www.inscc.utah.edu/~mani/cpex_cv/lifetime_prop/2022/09/

# #dimensions n_mcs = # of MCSs in the given month selected
#     #the MCSs are indexed in chronological order, with the associated MCS IDs 
#     #being found (in chronological order) at https://www.inscc.utah.edu/~mani/cpex_cv/TIMPS/2022/09/
# testds2 = xr.open_dataset(timps_test_path2)
# testds3 = xr.open_dataset(timps_test_path3)

# print ('MCS Propagation Speed (m/s):', testds2.propvecw_mag[1071].item())            #MCS ID 230041 (matches the MCS in testds above)
# print ('MCS Propagation Direction (deg, from):', testds3.propvecw_dir[1071].item())  #MCS ID 230041 (matches the MCS in testds above)
#     #How do we know that index 1071 matches MCS ID 230041? The data in 'propvecw_mag_Sept2022.nc' and 'propvecw_dir_Sept2022.nc' is not
#         #labeled by the MCS ID (sigh), but is given in the order that the MCS IDs show up on https://www.inscc.utah.edu/~mani/cpex_cv/TIMPS/2022/09/.
#         #So, copy/paste all the text from https://www.inscc.utah.edu/~mani/cpex_cv/TIMPS/2022/09/ into an Excel spreadsheet, delete the
#         #header rows so that just the MCS ID rows remain, then search for the desired MCS ID and note its Excel row number.
#         #The index for the desired MCS ID in 'propvecw_mag_Sept2022.nc' and 'propvecw_dir_Sept2022.nc' will be that row number MINUS 1 (because Python is a 0-based indexing system)

#         #There is probably an easier way to find the index from https://www.inscc.utah.edu/~mani/cpex_cv/TIMPS/2022/09/ using
#             #Python web scraping, but that requires more in-depth coding that I felt would take more time than the manual way described above

# testds2


In [ ]:
#get distance traveled (in km) and mean direction storm is coming from (see "Course 2-1" in http://edwilliams.org/gccalc.htm) from http://edwilliams.org/gccalc.htm
    #calculate mean storm propagation speed using (distance * 1000) / (3600 * storm timespan (in hours))
#^^^THIS WAY IS MUCH EASIER AND GIVES THE SAME RESULT AS THE MORE MANUAL VERSION BELOW

distance0 = 80.3712        #kilometers (see "Distance" output in http://edwilliams.org/gccalc.htm)
direction_from = 47.5567   #degrees (from) (see "Course 2-1" output in http://edwilliams.org/gccalc.htm)
timespan0 = 4              #hrs (see "Time Between Start/End Coordinates [hrs]" column in Sonde_Metric_Calculations_CAMP2Ex.csv)

distance = distance0 * 1000   #meters
timespan = timespan0 * 3600   #seconds
print ('Storm Distance Traveled (km):', np.round(distance0, 1))
print ('Storm Mean Propagation Speed (m/s):', np.round((distance / timespan), 1))      #propagation speed (m/s)
print ('Storm Mean Propagation Direction (deg, from):', np.round(direction_from, 1))   #propagation direction (deg, from)
print ('')

# ###########################################################################################################################################

# #OR, use this more manual version (more work though and still uses http://edwilliams.org/gccalc.htm)
#     #To calculate mean storm propagation speed and direction:
#         #1. Calculate the vector (components) connecting the weighted centroids of the storm start and end coordinates
#             #(i.e., delta lon/x and delta lat/y and convert to meters) 
#         #2. Calculate storm propagation speed/magnitude using distance / time = sqrt(deltaX^2 + deltaY^2) / storm timespan
#         #3. Calculate storm propagation direction (coming FROM) using np.degrees(np.arctan2(-deltax, -deltay))

# #get deltax (deltay) below from http://edwilliams.org/gccalc.htm by holding latitude (longitude) constant while calculating deltax (deltay)
#     #when calculating deltax using http://edwilliams.org/gccalc.htm, input the same latitude for Lat1 and Lat2 that is the mean latitude of the storm #(i.e., (start_lat + end_lat) / 2)
# deltax = -59.3501e3   #should be positive (negative) if the storm moves east (west) with time (meters)
# deltay = -54.1952e3   #should be positive (negative) if the storm moves north (south) with time (meters)
# timespan = 3600 * 4   #storm timespan between the start and end coordinates (seconds)

# direction_from = np.degrees(np.arctan2(-deltax, -deltay))   #(-x, -y)
# if direction_from < 0:
#     direction_from += 360
# distance = np.sqrt(deltax**2 + deltay**2)

# print ('Storm Distance Traveled (km):', np.round((distance / 1000), 1))
# print ('Storm Mean Propagation Speed (m/s):', np.round((distance / timespan), 1))      #propagation speed (m/s)
# print ('Storm Mean Propagation Direction (deg, from):', np.round(direction_from, 1))   #propagation direction (deg, from)

# #for testing: both methods should give distance traveled of 80.4 km, speed of 5.6 m/s, and direction of 47.6 degrees (from) for the following coordinates and timespan:
#     #start_lat = 9.41
#     #start_lon = 118.87
#     #end_lat = 8.92
#     #end_lon = 118.33
#     #timespan = 4


In [ ]:
###the only variables you (Ben) need to change for the rest of this script are the 3 in this cell.
    ###For other users, you will need to change much more, including filepaths and downloading IMERG/TIMPS data (see link in header)

plot_timps_sonde_srm = True          #determines whether to calculate sonde storm relative motion (SRM) or not
SRM_layers = ['Low', 'Mid', 'Nah']   #low-level (975-925 hPa) and mid-level (900-700 hPa) layers for which to calculate sonde SRM vectors
#SRM_layers = ['Nah']                #low-level (975-925 hPa) and mid-level (900-700 hPa) layers for which to calculate sonde SRM vectors

#CAMP2Ex convective case times, per CAMP2Ex Well Documented Convection.docx and IRCOLOR_JPL_Portal/ folder for each flight date
    #key: start date of the flight time range
    #item: list of two lists:
        #item[0]: list of hours to be plotted for the given flight in CHRONOLOGICAL order (this is important to assign case_date correctly)
        #item[1]: list of the lat/lon extent [West,East,South,North] for plotting the desired flight at the given time and for associated TIMPS ID filtering
case_dict = {
             '20190829': [[21, 22, 23, 0, 1, 2, 3, 4, 5, 6, 7], [115, 130, 5, 20]],
             '20190831': [[0, 1, 2, 3, 4, 5, 6, 7, 8], [112.5, 127.5, 5, 20]],
             '20190904': [[0, 1, 2, 3, 4, 5, 6, 7, 8, 9], [112.5, 127.5, 2.5, 17.5]],
             '20190907': [[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10], [112.5, 127.5, 7.5, 22.5]],
             '20190909': [[0, 1, 2, 3, 4, 5], [115, 130, 10, 25]],
             '20190915': [[15, 16, 17, 18, 19, 20, 21, 22, 23, 0, 1, 2, 3, 4, 5, 6, 7], [112.5, 127.5, 2.5, 17.5]],
             '20190917': [[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17], [117.5, 132.5, 10, 25]],
             '20191001': [[18, 19, 20, 21, 22, 23, 0, 1, 2, 3], [112.5, 127.5, 10, 25]],
             '20191002': [[4, 5, 6, 7, 8, 9, 10, 11, 12], [112.5, 127.5, 10, 25]],
             '20191003': [[21, 22, 23, 0, 1, 2, 3, 4, 5, 6], [115, 130, 7.5, 22.5]],
             '20191004': [[0, 1, 2, 3, 4, 5, 6], [115, 130, 7.5, 22.5]],
             '20191005': [[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14], [115, 130, 10, 25]]
            }

# #for testing
#case_dict = {'20190917': [3]}   #existing TIMPS ID and dropsonde GPS/wind data (as opposed to the 2 cases below)
#case_dict = {'20190830': [2]}   #for testing (has N/A TIMPS IDs)
#case_dict = {'20191005': [6]}   #for testing (has dropsondes with no GPS/winds)


In [ ]:
#This cell is used for determining the TIMPS ID for each CAMP2Ex Organized TOC case using TIMPS_data/ folder files,
#and is subsequently used to make plots of GPM IMERG, TIMPS MCS ID weighted centroids, and CAMP2Ex flight tracks
#with sonde locations and/or calculated sonde low/mid-level SRM vectors overlaid

tstart = time.time()

timps_folder = os.path.join(os.getcwd(), 'TIMPS_data')
timps_files_folder = os.path.join(timps_folder, 'TIMPS_files')
sonde_metric_filepath = os.path.join(os.getcwd(), 'Sonde_Metric_Calculations_CAMP2Ex.csv')

#set some baseline plot displays
#matplotlib.rcParams['axes.facecolor'] = [0.9,0.9,0.9]
matplotlib.rcParams['axes.labelsize'] = 18
matplotlib.rcParams['axes.titlesize'] = 18
matplotlib.rcParams['axes.labelweight'] = 'bold'
matplotlib.rcParams['axes.titleweight'] = 'bold'
matplotlib.rcParams['xtick.labelsize'] = 16
matplotlib.rcParams['ytick.labelsize'] = 16
matplotlib.rcParams['legend.fontsize'] = 16
#matplotlib.rcParams['legend.facecolor'] = 'w'
#matplotlib.rcParams['axes.facecolor'] = 'w'
matplotlib.rcParams['font.family'] = 'arial'
matplotlib.rcParams['hatch.linewidth'] = 0.3


for SRM_layer in SRM_layers:

    #case_date is the date on which the desired flight took place
    #case_ranges[0] is a list of the desired UTC hours to be plotted for the given flight (e.g., convective case hours/time range)
    #case_ranges[1] is a list of the lat/lon extent [West,East,South,North] for plotting the desired flight at the given time and for associated TIMPS ID filtering
    for case_date, case_ranges in case_dict.items():

        print (f'{case_date} {SRM_layer}-level SRM plotting in progress...')
    
        case_date_original = case_date + ''   #same as case_date, but adding '' just to make sure case_date_original doesn't change if case_date is later changed (potential Python variable copying issue)
        
        plot_save_folder = os.path.join(os.getcwd(), case_date_original, 'IMERG_TIMPS_IDs')
        campaign_extent = case_ranges[1]     #lat/lon extent [West,East,South,North] for plotting the desired flight at the given time and for associated TIMPS ID filtering
    
        #Navigation data
        nav_folder = os.path.join(os.getcwd(), case_date_original, 'Nav_files')
        for x in os.listdir(nav_folder):
            if x[0:3] == '.DS':         #delete hidden .DS_Store files if they come up (will show up if you delete a file)
                os.remove(os.path.join(nav_folder, x))
        nav_ict_path = os.path.join(nav_folder, os.listdir(nav_folder)[0])  #only one nav file per flight
        nav_ict = icartt.Dataset(nav_ict_path)    #open the ict file with icartt
        flight_lat = nav_ict.data["Latitude"]     #nav latitude, just a normal 1-D array
        flight_lon = nav_ict.data["Longitude"]    #nav longitude, just a normal 1-D array
    
        #load the final dropsonde CSV
        drop_csv_path = os.path.join(os.getcwd(), case_date_original, 'final_dropsonde_' + case_date_original + '.csv')
        drop_csv = pd.read_csv(drop_csv_path)

        if case_ranges[0][-1] < case_ranges[0][0]:   #include the following day's sonde CSV (if it exists) if the flight goes into the following day
                                                     #used for flights that go into the following day but have cases spanning only one of the days (i.e., 20191001 and 20191003)
            next_date = datetime.strftime(datetime.strptime(case_date_original, '%Y%m%d') + timedelta(days = 1), '%Y%m%d')
            drop_csv_path2 = os.path.join(os.getcwd(), next_date, 'final_dropsonde_' + next_date + '.csv')
            if os.path.isfile(drop_csv_path2):       #most flights that go into the following day don't have cases spanning only one of the days and thus won't have a data folder for the next day
                drop_csv = pd.concat([drop_csv, pd.read_csv(drop_csv_path2)], ignore_index = True)  #concatenates fields with same header
            else:
                pass
    
        #load the final radiosonde CSV and add concatenate onto drop_csv (if applicable)
        radio_csv_path = os.path.join(os.getcwd(), case_date_original, 'final_radiosonde_' + case_date_original + '.csv')
        if os.path.isfile(radio_csv_path):          
            sonde_csv = pd.concat([drop_csv, pd.read_csv(radio_csv_path)], ignore_index = True)  #concatenates fields with same header
        else:
            sonde_csv = drop_csv.copy()
    
        df_sonde = pd.read_csv(sonde_metric_filepath)
    
        if case_ranges[0][-1] < case_ranges[0][0]:   #include the following day's sondes if the flight goes into the following day
            df_sonde_use = df_sonde[(df_sonde['Date'] == int(case_date_original)) | (df_sonde['Date'] == int(datetime.strftime(datetime.strptime(case_date_original, '%Y%m%d') + timedelta(days = 1), '%Y%m%d')))].copy()
        else:
            df_sonde_use = df_sonde[df_sonde['Date'] == int(case_date_original)].copy()
    
        if plot_timps_sonde_srm:
            sonde_data_to_plot = np.empty((len(df_sonde_use), 7), dtype = object)
        else:
            sonde_data_to_plot = np.empty((len(df_sonde_use), 3), dtype = object)
    
        for x in range(len(df_sonde_use)):
            
            date = str(df_sonde_use['Date'].iloc[x])
            time0 = str(df_sonde_use['Time'].iloc[x]).zfill(6)
            sonde_datetime = date[:4] + '-' + date[4:6] + '-' + date[6:] + ' ' + time0[:2] + ':' + time0[2:4] + ':' + time0[4:]
    
            #calculate the sonde's mean lat/lon (no lats/lons if the sonde doesn't have any wind data)
            sonde_csv_use = sonde_csv[sonde_csv['Time [UTC]'] == sonde_datetime].copy()
            sonde_mean_lon = sonde_csv_use['Longitude [deg]'].mean()
            sonde_mean_lat = sonde_csv_use['Latitude [deg]'].mean()
            
            if pd.isna(sonde_mean_lon) or pd.isna(sonde_mean_lat):
                print (f'Skipping {sonde_datetime} sonde plotting due to no GPS data (and thus no wind data)')
                continue

            sonde_data_to_plot[x, :3] = [sonde_mean_lon, sonde_mean_lat, sonde_datetime]
    
            if plot_timps_sonde_srm:
                
                # #skip sondes without a TIMPS ID (i.e., dont add these sondes to sonde_data_to_plot)
                # if pd.isna(df_sonde_use['TIMPS ID'].iloc[x]):
                #     print (f'Skipping {sonde_datetime} sonde due to TIMPS ID = N/A')
                #     continue
                
                if df_sonde_use['Environment Falling In'].iloc[x] == 'In Precip':
                    print (f'Skipping {sonde_datetime} sonde TIMPS SRM plotting due to it being "In Precip" (In Precip sondes cannot be inflow sondes)')
                    continue
                
                #convert TIMPS mean propagation speed and direction to vector components
                TIMPS_spd = df_sonde_use['TIMPS ID Propagation Speed [m/s]'].iloc[x] * units('m/s')
                TIMPS_dir = df_sonde_use['TIMPS ID Propagation Direction (from) [deg]'].iloc[x] * units.deg
                TIMPS_u, TIMPS_v = mpcalc.wind_components(TIMPS_spd, TIMPS_dir)  #returns u,v values in whatever unit TIMPS_spd is in
                TIMPS_u = TIMPS_u.m  #get rid of the Pint units
                TIMPS_v = TIMPS_v.m  #get rid of the Pint units
                
                #calculate the sonde's mean-layer low-level wind components, along with mean-layer low-level SRMs
                sonde_csv_low = sonde_csv_use[(sonde_csv_use['Pressure [mb]'] <= 975) & (sonde_csv_use['Pressure [mb]'] >= 925)].copy()
                sonde_mean_u_low = sonde_csv_low['U Comp of Wind [m/s]'].mean()
                sonde_mean_v_low = sonde_csv_low['V Comp of Wind [m/s]'].mean()
                
                if pd.isna(sonde_mean_u_low) or pd.isna(sonde_mean_v_low):
                    SRM_low_u = None
                    SRM_low_v = None
                else:   #SRM = sonde mean wind vector minus convective object (CO) motion vector
                    SRM_low_u = sonde_mean_u_low - TIMPS_u
                    SRM_low_v = sonde_mean_v_low - TIMPS_v
                
                #calculate the sonde's mean-layer mid-level wind components, along with mean-layer mid-level SRMs
                sonde_csv_mid = sonde_csv_use[(sonde_csv_use['Pressure [mb]'] <= 900) & (sonde_csv_use['Pressure [mb]'] >= 700)].copy()
                sonde_mean_u_mid = sonde_csv_mid['U Comp of Wind [m/s]'].mean()
                sonde_mean_v_mid = sonde_csv_mid['V Comp of Wind [m/s]'].mean()
        
                if pd.isna(sonde_mean_u_mid) or pd.isna(sonde_mean_v_mid):
                    SRM_mid_u = None
                    SRM_mid_v = None
                else:   #SRM = sonde mean wind vector minus convective object (CO) motion vector
                    SRM_mid_u = sonde_mean_u_mid - TIMPS_u
                    SRM_mid_v = sonde_mean_v_mid - TIMPS_v
                
                sonde_data_to_plot[x, 3:] = [SRM_low_u, SRM_low_v, SRM_mid_u, SRM_mid_v]
            else:
                pass
            
        #mask sonde rows (i.e., all values of None) that didn't have an associated TIMPS ID (may not be necessary with quiver plotting)
        #sonde_data_to_plot = np.ma.masked_where(sonde_data_to_plot == None, sonde_data_to_plot)
    
        for hr in case_ranges[0]:   #for each time (in UTC (HHMM)) for which the desired case should be plotted
    
            if hr < case_ranges[0][0]:   #update case_date accordingly if the flight goes into the following day
                add_day = datetime.strptime(case_date_original, '%Y%m%d') + timedelta(days = 1)
                case_date = datetime.strftime(add_day, '%Y%m%d')
            #print (case_date)   #a sanity check to make sure that the above 3 lines of code properly update case_date
    
            hr2 = str(hr).zfill(2)
            minute = 0
            minute2 = str(minute).zfill(2)
    
            timps_path = os.path.join(timps_folder, 'fields2D_precip_n_mcs', case_date[:-2], f'Tracking_{case_date + hr2 + minute2}.nc')
            ds = xr.open_dataset(timps_path)
            ds = ds.sel(lat = slice(campaign_extent[2], campaign_extent[3])).sel(lon = slice(campaign_extent[0], campaign_extent[1]))
            
            data_proj = ccrs.PlateCarree()
            
            group_fig = plt.figure(figsize = (16, 16))   #initialize the streamline figure for the given hour
            ax = group_fig.add_subplot(1, 1, 1, projection = data_proj)        
            
            precip = ds.Precipitation.sel(time = case_date)
            try:  #when time is 00:00 UTC, the fields2D_precip_n_mcs files do not contain time as an index for some odd reason (not your fault),
                  #so the time filter does not work even though the date filter does above.
                  #When this happens, ds.Precipitation.sel(time = case_date) gives a 2-D array (lat, lon) rather than a 3-D array, and since
                  #the file is already for just the specific wanted time, you just need to skip these next few lines for 00:00 UTC
                #precip = precip.sel(time = precip.time.dt.hour.isin(hr))[0]              #use the time on the hour (e.g., 03:00 UTC) and not the half hour (e.g., 03:30 UTC)
                precip = precip.sel(time = precip.time.dt.hour.isin(hr))
                precip = precip.sel(time = precip.time.dt.minute.isin(minute))[0]         #IMERG and TIMPS have 30-minute temporal resolution
            except KeyError:
                pass
            
            timps_id = ds.SystemID_MCS.sel(time = case_date)
            try:  #when time is 00:00 UTC, the fields2D_precip_n_mcs files do not contain time as an index for some odd reason (not your fault),
                  #so the time filter does not work even though the date filter does above.
                  #When this happens, ds.SystemID_MCS.sel(time = case_date) gives a 2-D array (lat, lon) rather than a 3-D array, and since
                  #the file is already for just the specific wanted time, you just need to skip these next few lines for 00:00 UTC
                #timps_id = timps_id.sel(time = timps_id.time.dt.hour.isin(hr))[0]        #use the time on the hour (e.g., 03:00 UTC) and not the half hour (e.g., 03:30 UTC)
                timps_id = timps_id.sel(time = timps_id.time.dt.hour.isin(hr))
                timps_id = timps_id.sel(time = timps_id.time.dt.minute.isin(minute))[0]   #IMERG and TIMPS have 30-minute temporal resolution
            except KeyError:
                pass
                
            timps_id_array = timps_id.values
            
            #find the unique TIMPS IDs for the given campaign_extent lat/lon range (ultimately to find the TIMPS ID associated with the organized TOC system of interest) 
            timps_unique_ids_list = list(pd.Series(timps_id_array.flatten()).unique())
            timps_unique_ids = []
            for xx in timps_unique_ids_list:
                if xx < 1:  #NaN values in ds.SystemID_MCS manifest as zeros (this line of code will also omit regular NaNs as well, if they exist)
                    continue
                else:
                    timps_unique_ids.append(str(int(xx))[4:])   #7-digit TIMPS IDs have a prefix of the year (YYYY) they were first identified, so getting rid of that year to avoid confusion
            #print (case_date, hr2, 'UTC unique TIMPS IDs within plot range:', timps_unique_ids)
            
            ######################################################################################################
                
            ax.set_extent(campaign_extent, ccrs.PlateCarree())   #lat/lon bounds are [West,East,South,North]
            
            # Add land, coastlines, and borders
            #ax.add_feature(cfeature.LAND, facecolor='0.8')
            ax.coastlines(ls = '-', linewidth = 1.5, color = 'k')
            
            #Gridlines
            gl = ax.gridlines(crs = ccrs.PlateCarree(), draw_labels = True, linewidth = 0.5, color = 'gray', alpha = 0.5, linestyle = '--')
            gl.top_labels = False
            gl.right_labels = False
            gl.xlabel_style = {'size':16, 'color':'black'}
            gl.ylabel_style = {'size':16, 'color':'black'}
            
            imerg_levels = np.arange(0, 100.1, 2)   #amount of contour levels to plot (not the actual values)
            
            #plot IMERG Rain Rate
            pm1 = ax.contourf(precip.lon, precip.lat, precip.values, 
                              levels = np.logspace(np.log10(0.1), np.log10(40), num = len(imerg_levels)), 
                              norm = 'log', extend = 'max', cmap = cm.jet, transform = data_proj, zorder = 0)
    
            #plot flight track
            ax.plot(flight_lon, flight_lat, color = 'darkmagenta', linewidth = 3.0, zorder = 1)
            
            # #plot TIMPS MCS IDs
            # #mask non-TIMPS IDs (values < 1 (11 digits per TIMPS MCS: each 7-digit TIMPS ID has a prefix of the year (YYYY) they were first identified))
            # timps_id_masked = np.ma.masked_where(timps_id_array < 1, timps_id_array)
            # timps_id_masked = np.ma.masked_where(np.isnan(timps_id_masked), timps_id_masked)  #masks NaN values (not masked in previous line)
            # pm2 = ax.pcolormesh(timps_id.lon, timps_id.lat, timps_id_masked, cmap = cm.Blues, transform = data_proj, zorder = 1)

            #plot TIMPS MCS ID centroids
            for unique_timps_id in timps_unique_ids:
                timps_filepath = None
                for filename in os.listdir(timps_files_folder):
                    if unique_timps_id in filename:
                        timps_filepath = os.path.join(timps_files_folder, filename)
                        break
                
                if timps_filepath == None:
                    #sys.exit(f'Could not find TIMPS file for TIMPS ID {unique_timps_id}')
                    print (f'Could not find TIMPS file to plot TIMPS ID {unique_timps_id} for {case_date} at {hr2} UTC')
                else:
                    timps_ds = xr.open_dataset(timps_filepath)
                    timps_ds = timps_ds.sel(time = case_date)
                    #timps_ds = timps_ds.sel(time = timps_ds.time.dt.hour.isin(hr))[0]       #use the time on the hour (e.g., 03:00 UTC) and not the half hour (e.g., 03:30 UTC)
                    timps_ds = timps_ds.sel(time = timps_ds.time.dt.hour.isin(hr))
                    timps_ds = timps_ds.sel(time = timps_ds.time.dt.minute.isin(minute))     #IMERG and TIMPS have 30-minute temporal resolution
                    timps_clonw = timps_ds.centlonwgt.values[0]                              #longitude of the TIMPS MCS weighted centroid
                    timps_clatw = timps_ds.centlatwgt.values[0]                              #latitude of the TIMPS MCS weighted centroid
            
                    if (timps_clonw >= campaign_extent[0]) and (timps_clonw <= campaign_extent[1]) and (timps_clatw >= campaign_extent[2]) and (timps_clatw <= campaign_extent[3]):
                        ax.scatter(timps_clonw, timps_clatw, marker = 'o', color = 'k', s = 120, zorder = 2)
                        ax.text(timps_clonw, timps_clatw, unique_timps_id, color = 'k', fontsize = 'xx-large', ha = 'left', va = 'bottom', zorder = 3)
                    
                    timps_ds.close()
    
            if plot_timps_sonde_srm and SRM_layer == 'Low':            #plot low-level sonde SRM vector
                for sonde_row in range(sonde_data_to_plot.shape[0]):
                    if sonde_data_to_plot[sonde_row, 3] != None:       #In Precip sonde rows 4-7 will be all None values when plot_timps_sonde_srm = True
                                                                       #Using row 4 here also, in case SRM_low_u/SRM_low_v could not be calculated for some reason
                                                                       #sonde_data_to_plot[sonde_row, 0] != None is also implied here with this Boolean condition
                        if (datetime.strptime(sonde_data_to_plot[sonde_row, 2], '%Y-%m-%d %H:%M:%S') <= (datetime.strptime(case_date + hr2 + minute2, '%Y%m%d%H%M') + timedelta(minutes = 30))) and (datetime.strptime(sonde_data_to_plot[sonde_row, 2], '%Y-%m-%d %H:%M:%S') >= (datetime.strptime(case_date + hr2 + minute2, '%Y%m%d%H%M') - timedelta(minutes = 30))):
                            ax.quiver(sonde_data_to_plot[sonde_row, 0].astype(float), sonde_data_to_plot[sonde_row, 1].astype(float), sonde_data_to_plot[sonde_row, 3].astype(float), sonde_data_to_plot[sonde_row, 4].astype(float), color = 'k', pivot = 'middle', zorder = 4)
                ax.set_title('GPM IMERG, TIMPS MCS ID Weighted Centroids, and CAMP2Ex Flight Track\nwith Sonde Low-level (975-925 hPa) SRM Vectors (%s, %s UTC)' % (case_date, hr2))
                plot_save_name = f'{case_date}_{hr2}UTC_IMERG_TIMPS_MCS_ID_Centroids_FlightTrack_Sondes_LowLevel_SRMs.png'
            elif plot_timps_sonde_srm and SRM_layer == 'Mid':          #plot mid-level sonde SRM vector
                for sonde_row in range(sonde_data_to_plot.shape[0]):
                    if sonde_data_to_plot[sonde_row, 5] != None:       #In Precip sonde rows 4-7 will be all None values when plot_timps_sonde_srm = True
                                                                       #Using row 6 here also, in case SRM_mid_u/SRM_mid_v could not be calculated for some reason
                                                                       #sonde_data_to_plot[sonde_row, 0] != None is also implied here with this Boolean condition
                        if (datetime.strptime(sonde_data_to_plot[sonde_row, 2], '%Y-%m-%d %H:%M:%S') <= (datetime.strptime(case_date + hr2 + minute2, '%Y%m%d%H%M') + timedelta(minutes = 30))) and (datetime.strptime(sonde_data_to_plot[sonde_row, 2], '%Y-%m-%d %H:%M:%S') >= (datetime.strptime(case_date + hr2 + minute2, '%Y%m%d%H%M') - timedelta(minutes = 30))):
                            ax.quiver(sonde_data_to_plot[sonde_row, 0].astype(float), sonde_data_to_plot[sonde_row, 1].astype(float), sonde_data_to_plot[sonde_row, 5].astype(float), sonde_data_to_plot[sonde_row, 6].astype(float), color = 'k', pivot = 'middle', zorder = 4)
                ax.set_title('GPM IMERG, TIMPS MCS ID Weighted Centroids, and CAMP2Ex Flight Track\nwith Sonde Mid-level (900-700 hPa) SRM Vectors (%s, %s UTC)' % (case_date, hr2))
                plot_save_name = f'{case_date}_{hr2}UTC_IMERG_TIMPS_MCS_ID_Centroids_FlightTrack_Sondes_MidLevel_SRMs.png'
            else:
                #plot near-storm sonde locations for the given flight if the sonde was deployed within 30 minutes (1-hr total range) of the given hour
                        #NOTE: Sondes with no wind data don't have GPS data either (1 of them total)
                for sonde_row in range(sonde_data_to_plot.shape[0]):
                    if sonde_data_to_plot[sonde_row, 0] != None:           #a sonde without GPS/wind data will have a row with all None values
                        if (datetime.strptime(sonde_data_to_plot[sonde_row, 2], '%Y-%m-%d %H:%M:%S') <= (datetime.strptime(case_date + hr2 + minute2, '%Y%m%d%H%M') + timedelta(minutes = 30))) and (datetime.strptime(sonde_data_to_plot[sonde_row, 2], '%Y-%m-%d %H:%M:%S') >= (datetime.strptime(case_date + hr2 + minute2, '%Y%m%d%H%M') - timedelta(minutes = 30))):
                            ax.scatter(sonde_data_to_plot[sonde_row, 0], sonde_data_to_plot[sonde_row, 1], marker = '*', color = 'k', s = 250, zorder = 5)
                            ax.text(sonde_data_to_plot[sonde_row, 0], sonde_data_to_plot[sonde_row, 1], sonde_data_to_plot[sonde_row, 2][11:16], color = 'k', fontsize = 'xx-large', ha = 'center', va = 'top', zorder = 5)
                ax.set_title('GPM IMERG, TIMPS MCS ID Weighted Centroids, and CAMP2Ex Flight Track\nwith Sonde Locations (%s, %s UTC)' % (case_date, hr2))
                plot_save_name = f'{case_date}_{hr2}UTC_IMERG_TIMPS_MCS_ID_Centroids_FlightTrack_Sondes.png'
            
            #IMERG colorbar
            ticks_imerg = np.array([0.1, 1, 5, 10, 20, 40], dtype = float)
            cax1 = group_fig.add_axes([ax.get_position().x1 + 0.03, ax.get_position().y0, 0.02, 0.77])
            cbar1 = group_fig.colorbar(pm1, cax = cax1, ticks = ticks_imerg)
            #cbar1 = group_fig.colorbar(pm1, ax = ax, ticks = ticks_imerg)
            cbar1.set_label('IMERG [mm hr$\\bf{^{-1}}$]')
            cbar1.ax.set_yticklabels(list(map(str, list(ticks_imerg))))  #labels automatically default to tick values given to ticks parameter in fig.colorbar(), unless you're using a log scale I guess
            cbar1.ax.yaxis.set_ticks_position('right')
            cbar1.ax.yaxis.set_label_position('right')               
                    
            #plt.tight_layout()
            #plt.subplots_adjust(wspace = 0.1)
            
            #save the figure
            plt.savefig(os.path.join(plot_save_folder, plot_save_name), bbox_inches = 'tight')
            #plt.show()  #plt.show() must come after plt.savefig() in order for the image to save properly
            #plt.clf()   #supposedly speeds things up? According to: https://www.youtube.com/watch?v=jGVIZbi9uMY
            plt.close()
            plt.clf()    #if placing this after plt.close(), may release memory related to the figure (https://stackoverflow.com/questions/741877/how-do-i-tell-matplotlib-that-i-am-done-with-a-plot)
            
            ##decrease file size of the image by 66% without noticeable image effects (if using Matplotlib)
            ##(good to use if you're producing a lot of images, see https://www.youtube.com/watch?v=fzhAseXp5B4)
            im = Image.open(os.path.join(plot_save_folder, plot_save_name))
            
            try:
                im2 = im.convert('P', palette = Image.Palette.ADAPTIVE)
            except:
                #use this for older version of PIL/Pillow if the above line doesn't work, 
                #though this line will have isolated, extremely minor image effects due to 
                #only using 256 colors instead of the 3-element RGB scale
                im2 = im.convert('P')
            
            im2.save(os.path.join(plot_save_folder, plot_save_name))
            im.close()
            im2.close()
            
            ds.close()

        print (f'{case_date_original} {SRM_layer}-level SRM plots complete!\n')

tend = time.time()
print (f'This script took {np.round((tend - tstart) / 60, 1)} minutes to complete.')
